<h1>PPDB TAP Examples</h1>

<h2>Setup</h2>

<h3>Imports</h3>

In [ ]:
import time

from lsst.rsp import get_tap_service

<h3>Initialize the TAP service</h3>

In [ ]:
service = get_tap_service("ppdbtap")

<h3>Query helper function</h3>

In [ ]:
def query(sql, print_info=True):
    """Run a query and return the results as a pandas Dataframe, 
    optionally printing the timing and row count.
    """
    if print_info:
        print(f"Executing query: {sql}")
    start = time.perf_counter()
    res = service.run_async(sql).resultstable
    end = time.perf_counter()
    df = res.to_table().to_pandas()
    if print_info:
        print(f"Query took {end - start:.3f} seconds and returned {len(df)} rows")
    return df

<h2>Example Queries</h2>

<h3>TAP Schema</h3>

In [ ]:
query("SELECT * FROM tap_schema.columns WHERE table_name LIKE 'ppdb.%'")

<h3>Find by ID</h3>

In [ ]:
query("SELECT * FROM ppdb.DiaObject WHERE diaObjectId=169760231406961662")

<h3>Cone search</h3>

Sample ra and dec

In [ ]:
ra = 250.0
dec = -20.0

In [ ]:
query(f"""
SELECT *
FROM ppdb.DiaObject
WHERE 
  validityEndMjdTai IS NULL AND 
  CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', {ra}, {dec}, 1.0)) = 1
""")

<h3>Nearest neighbor search</h3>

Nearest neighbors in the conical region within 1 arcminute.

In [ ]:
query(f"""
SELECT 
    o1.ra as ra1,
    o1.dec as dec1,
    o2.ra as ra2,
    o2.dec as dec2,
    o1.diaObjectId AS id1,
    o2.diaObjectId AS id2,
    DISTANCE(POINT('ICRS', o1.ra, o1.dec), POINT('ICRS', o2.ra, o2.dec)) AS dist
FROM ppdb.DiaObject AS o1
JOIN ppdb.DiaObject AS o2
  ON o1.diaObjectId <> o2.diaObjectId
  AND o1.validityEndMjdTai IS NULL
  AND o2.validityEndMjdTai IS NULL
WHERE CONTAINS(POINT('ICRS', o1.ra, o1.dec),
               CIRCLE('ICRS', {ra}, {dec}, 0.1)) = 1
  AND DISTANCE(POINT('ICRS', o1.ra, o1.dec),
               POINT('ICRS', o2.ra, o2.dec)) < 0.0166667;
""")

<h3>Table join</h3>

In [ ]:
query("""SELECT * FROM ppdb.DiaSource ds 
LEFT JOIN ppdb.DiaObject dob ON dob.diaObjectId = ds.diaObjectId
WHERE dob.diaObjectId=169760231406961662""")

<h3>Table scan</h3>

In [ ]:
query("""
SELECT * FROM ppdb.DiaObject 
WHERE r_psfFluxMean BETWEEN 1090.0 and 1100.0
""")

<h3>Row counts by day</h3>

In [ ]:
query("""
SELECT FLOOR(validityStartMjdTai) as mjd_tai_day, COUNT(*) as dia_objects
FROM ppdb.DiaObject
GROUP BY mjd_tai_day
ORDER BY mjd_tai_day DESC
""")